# 뉴스 수집 DB 조회

`news_collector.ipynb`가 MySQL(`news` DB)에 쌓은 결과를 확인하는 노트북이다. **조회만 한다.** DB를 바꾸는 코드는 없다.

**접속 주소**: 프로젝트 루트(또는 `graph_db`)의 `.env`에서 아래 순서로 찾는다.

| 키 | 언제 쓰나 |
|---|---|
| `NEWS_DB_READ_URL` | 다른 컴퓨터에서 읽기 전용 계정(`reader`)으로 볼 때. 예: `mysql+pymysql://reader:비밀번호@윈도우PC_IP:3306/news?charset=utf8mb4` |
| `NEWS_DB_URL` | 위 키가 없을 때. 수집기가 도는 윈도우 PC에서는 이 값(수집기 계정)을 그대로 쓴다 |

필요한 패키지: `pandas`, `sqlalchemy`, `pymysql`, `cryptography`

시각은 DB에 UTC로 저장되어 있고, 이 노트북에서는 **한국 시간(KST)** 으로 바꿔 보여 준다.

**한눈에**

| | |
|---|---|
| 처리 순서에서 | 처리 단계가 아니다. `news_collector`가 잘 돌고 있는지 볼 때 연다 |
| 외부 요청·비용 | 없음. 윈도우 PC의 MySQL을 읽기만 한다(같은 네트워크에서만 접속된다) |
| 다시 실행 | 언제든 된다 |

## 1. 접속

In [ ]:
# ── 접속: .env의 DB 주소(읽기 전용 계정 우선) ──
import os
import json
import datetime as dt
from pathlib import Path

import pandas as pd
import sqlalchemy as sa

pd.set_option('display.max_colwidth', 80)   # 표에서 긴 글자는 80자까지만 보여 준다
pd.set_option('display.max_rows', 100)

ROOT = Path.cwd()   # graph_db 폴더에서 연다고 가정한다
KST = dt.timezone(dt.timedelta(hours=9))


def env_value(name):
    """환경변수 → 프로젝트 루트 .env → graph_db/.env 순서로 값을 찾는다. 없으면 None."""
    if os.environ.get(name):
        return os.environ[name]
    for env_file in (ROOT.parent / '.env', ROOT / '.env'):
        if not env_file.exists():
            continue
        for line in env_file.read_text(encoding='utf-8-sig').splitlines():
            key, _, value = line.partition('=')
            if key.strip() == name and value.strip():
                return value.strip().strip('"\'')
    return None


DB_URL = env_value('NEWS_DB_READ_URL') or env_value('NEWS_DB_URL')
assert DB_URL, '.env에 NEWS_DB_READ_URL 또는 NEWS_DB_URL이 없다.'

engine = sa.create_engine(DB_URL, pool_pre_ping=True, connect_args={'connect_timeout': 10})
print('접속 주소:', engine.url.render_as_string(hide_password=True))   # 비밀번호는 가린다

In [ ]:
# ── 조회 함수(SQL → 표, 시각은 KST로)와 접속 확인 ──
def query(sql, **params):
    """SQL을 실행해 결과를 DataFrame으로 돌려준다. 값은 :이름 자리에 params로 넘긴다(직접 문자열에 끼우지 않는다)."""
    with engine.connect() as c:
        return pd.read_sql(sa.text(sql), c, params=params)


def to_kst(series):
    """UTC ISO 문자열 열 → 한국 시간(분 단위) 문자열."""
    return (pd.to_datetime(series, utc=True, errors='coerce')
              .dt.tz_convert(KST).dt.strftime('%m-%d %H:%M'))


def utc_ago(hours):
    """지금부터 hours시간 전의 UTC 시각을 DB와 같은 ISO 문자열로. (DB의 시각 열은 문자열이라 문자열끼리 비교한다)"""
    return (dt.datetime.now(dt.timezone.utc) - dt.timedelta(hours=hours)).replace(microsecond=0).isoformat()


# 접속 확인: 계정 이름과 MySQL 버전. 실패하면 MySQL 오류 번호에 따라 원인과 해결 방법을 알려 준다
try:
    display(query('SELECT CURRENT_USER() AS 계정, VERSION() AS MySQL버전'))
except sa.exc.OperationalError as e:
    code = e.orig.args[0] if getattr(e, 'orig', None) is not None and e.orig.args else None
    host = engine.url.host
    if code == 1045:      # 계정은 찾았지만 비밀번호가 틀림
        hint = '비밀번호가 틀렸다. .env 주소의 "reader:비밀번호@" 부분을 확인한다.'
    elif code == 1044:    # 접속은 됐지만 news DB 권한이 없음
        hint = '계정에 news DB 권한이 없다. 윈도우 PC의 Workbench에서 GRANT SELECT ON news.* TO ... 를 확인한다.'
    elif code == 2003 and host in ('localhost', '127.0.0.1'):
        hint = ('주소가 localhost인데 이 컴퓨터에 MySQL이 없다. .env에 윈도우 PC의 읽기 전용 주소를 '
                'NEWS_DB_READ_URL=mysql+pymysql://reader:비밀번호@윈도우PC_IP:3306/news?charset=utf8mb4 로 넣고 커널을 다시 시작한다.')
    elif code == 2003:    # 서버에 닿지 않음
        hint = ('윈도우 PC에 닿지 않는다. ① 같은 와이파이(공유기)에 연결돼 있는지 ② 윈도우 PC의 IP가 바뀌지 않았는지(윈도우에서 ipconfig) '
                '③ 윈도우 PC가 켜져 있고 MySQL80 서비스가 돌고 있는지 확인한다.')
    elif 'cryptography' in str(e):
        hint = '패키지가 없다. %pip install cryptography 후 커널을 다시 시작한다.'
    else:
        hint = '아래 원래 오류 메시지를 확인한다.'
    raise RuntimeError(f'MySQL에 접속하지 못했다 ({host}). {hint}\n원래 오류: {e.orig}') from e

## 2. 수집기가 돌고 있는지

수집기는 언론사 피드를 1분마다, 전문지 피드를 15분마다 받는다. **마지막 피드 요청이 몇 분 전인지**를 보면 지금 돌고 있는지 알 수 있다. 몇 분 안이면 정상이고, 30분 넘게 비어 있으면 수집기나 윈도우 PC가 멈춘 것이다.

In [ ]:
# ── 2. 수집기가 돌고 있는지: 마지막 피드·본문 요청, 마지막 새 기사 ──
status = query("""
    SELECT MAX(checked_utc) AS 마지막_피드요청,
           (SELECT MAX(checked_utc) FROM body_log) AS 마지막_본문요청,
           (SELECT MAX(first_seen_utc) FROM articles) AS 마지막_새기사
    FROM fetch_log
""")
now = pd.Timestamp.now(tz='UTC')
for col in status.columns:
    t = pd.to_datetime(status.at[0, col], utc=True)
    minutes = (now - t).total_seconds() / 60 if pd.notna(t) else None
    when = t.tz_convert(KST).strftime('%m-%d %H:%M') if pd.notna(t) else '없음'
    print(f'{col:10s} {when} (KST)  →  {minutes:,.0f}분 전' if minutes is not None else f'{col:10s} 없음')

## 3. 표별 건수

In [ ]:
# ── 3. 표별 건수 ──
TABLES = {
    'articles':         '기사 (한 기사에 한 행)',
    'article_feeds':    '기사가 어느 피드에서 나왔는지',
    'article_versions': 'RSS에서 내용이 바뀐 기사의 이전 내용',
    'feeds':            '피드 목록',
    'feed_state':       '피드별 수집 상태',
    'fetch_log':        '피드 요청 기록',
    'body_log':         '기사 페이지 요청 기록',
}
counts = pd.DataFrame(
    [(name, desc, query(f'SELECT COUNT(*) AS n FROM {name}').at[0, 'n']) for name, desc in TABLES.items()],
    columns=['표', '내용', '건수'])
counts

## 4. 새 기사 수

- 첫 번째 표: 최근 24시간 동안 **한 시간마다** 들어온 새 기사 수
- 두 번째 표: 매체별 기사 수. 최근 24시간과 전체를 나란히 보여 준다. `tier`는 `core`(일반 경제 언론사)·`trade`(업종 전문지)

In [ ]:
# ── 4. 최근 24시간 시간별 새 기사 ──
recent = query('SELECT first_seen_utc FROM articles WHERE first_seen_utc >= :since', since=utc_ago(24))
recent['시간(KST)'] = pd.to_datetime(recent['first_seen_utc'], utc=True).dt.tz_convert(KST).dt.strftime('%m-%d %H시')
recent.groupby('시간(KST)').size().rename('새 기사').to_frame()

In [ ]:
# ── 4. 매체별 기사 수 ──
# 한 매체에 피드가 여러 개(CNBC 9개 등)라서, 피드 정보를 매체별로 먼저 하나로 묶은 뒤 연결한다
# (그냥 연결하면 같은 기사가 피드 수만큼 여러 번 세어진다)
by_source = query("""
    SELECT a.source                                         AS 매체,
           f.tier,
           f.sector                                         AS 업종,
           SUM(CASE WHEN a.first_seen_utc >= :since THEN 1 ELSE 0 END) AS 최근24시간,
           COUNT(*)                                         AS 전체
    FROM articles a
    LEFT JOIN (SELECT source, MAX(tier) AS tier, MAX(sector) AS sector
               FROM feeds GROUP BY source) f ON f.source = a.source
    GROUP BY a.source, f.tier, f.sector
    ORDER BY 최근24시간 DESC, 전체 DESC
""", since=utc_ago(24))
by_source[['최근24시간', '전체']] = by_source[['최근24시간', '전체']].astype(int)
by_source

## 5. 본문 상태

| 상태 | 뜻 |
|---|---|
| `ok` | 본문을 받았고 점검 통과 |
| `pending` | 아직 페이지를 열지 않음(같은 서버는 60초 간격이라 차례를 기다리는 중) |
| `short` | 150단어 미만. 원래 짧은 기사가 많다 |
| `suspect` | 제목과 내용이 안 맞거나 너무 많이 지움 |
| `paywalled`, `blocked` | 유료벽, 봇 차단. 다시 요청하지 않는다 |
| `sponsored`, `skipped_rule` | 후원 기사, 주소 규칙으로 페이지를 열지 않은 기사 |
| `gone`, `expired`, `error`, `robots`, `empty` | 주소 없음, 3일 안에 못 받음, 반복 오류, robots.txt 금지, 본문을 못 뽑음 |

In [ ]:
# ── 5. 본문 상태(전체) ──
# 전체 본문 상태
query("""
    SELECT COALESCE(body_status, '(없음)') AS 상태, COUNT(*) AS 건수
    FROM articles GROUP BY 1 ORDER BY 2 DESC
""")

In [ ]:
# ── 5. 본문 상태(매체별) ──
# 매체별 본문 상태: ok 비율이 낮은 매체가 위로 온다
bodies = query("SELECT source AS 매체, COALESCE(body_status, '(없음)') AS 상태 FROM articles")
table = bodies.pivot_table(index='매체', columns='상태', aggfunc='size', fill_value=0)
table['기사'] = table.sum(axis=1)
table['ok 비율'] = (table.get('ok', 0) / table['기사']).round(2)
table.sort_values('ok 비율')

## 6. 최근 기사

In [ ]:
# ── 6. 최근 기사 ──
LATEST_N = 20   # 몇 건을 볼지

latest = query("""
    SELECT first_seen_utc, source, body_status, body_words, title, url, article_id
    FROM articles
    ORDER BY first_seen_utc DESC
    LIMIT :n
""", n=LATEST_N)
latest.insert(0, '수집(KST)', to_kst(latest.pop('first_seen_utc')))
latest

## 7. 기사 하나 자세히 보기

`show_article(article_id)`: 정제한 본문, 정제에서 지운 줄, 점검 결과를 보여 준다. `article_id`는 위 표의 마지막 열에서 복사한다. 인자 없이 부르면 가장 최근에 본문을 받은 기사를 보여 준다.

In [ ]:
# ── 7. 기사 하나 자세히 보기 ──
def show_article(article_id=None, chars=1500):
    """기사 한 건의 제목·주소·본문(앞 chars자)·지운 줄·점검 결과를 출력한다."""
    if article_id is None:   # 가장 최근에 본문을 받은 기사
        sql = "SELECT * FROM articles WHERE body_status = 'ok' ORDER BY body_checked_utc DESC LIMIT 1"
        rows = query(sql)
    else:
        rows = query('SELECT * FROM articles WHERE article_id = :aid', aid=article_id)
    if rows.empty:
        print('해당 기사가 없다.')
        return
    a = rows.iloc[0]

    print(f"[{a['source']}] {a['title']}")
    print(a['url'])
    print(f"발행 {a['published_utc']} | 수집 {a['first_seen_utc']} (UTC)")
    print(f"본문 상태: {a['body_status']} ({a['body_source']}) | {a['body_words']}단어")
    print('-' * 100)
    text = a['body_text'] or ''
    print(text[:chars] + (' …' if len(text) > chars else ''))

    removed = json.loads(a['body_removed'] or '[]')   # [[규칙, 지운 줄], ...]
    if removed:
        print('-' * 100)
        print(f'정제에서 지운 줄 {len(removed)}개')
        for rule, line in removed:
            print(f'  [{rule}] {line[:120]}')
    if a['body_flags']:
        print('-' * 100)
        print('점검 결과:', a['body_flags'])


show_article()

## 8. 키워드 검색

제목이나 본문에 키워드가 들어간 기사를 찾는다. 예: 국내 기업 이름(`Samsung`, `SK Hynix`, `Hyundai`), 제품(`HBM`, `LNG carrier`)

In [ ]:
# ── 8. 키워드 검색 ──
def search(keyword, n=20, in_body=True):
    """제목(in_body=True면 본문도)에 keyword가 들어간 기사를 최근 순으로 n건 보여 준다. 대소문자는 가리지 않는다."""
    where = 'title LIKE :kw' + (' OR body_text LIKE :kw' if in_body else '')
    found = query(f"""
        SELECT first_seen_utc, source, body_status, title, url, article_id
        FROM articles
        WHERE {where}
        ORDER BY first_seen_utc DESC
        LIMIT :n
    """, kw=f'%{keyword}%', n=n)
    found.insert(0, '수집(KST)', to_kst(found.pop('first_seen_utc')))
    print(f"'{keyword}' 검색 결과 {len(found)}건 (최대 {n}건)")
    return found


search('Samsung')

In [ ]:
# ── 예: SK가 들어간 기사 ──
df_SK = search('SK')
_=df_SK['article_id'].apply(show_article, chars=500)

In [ ]:
# ── 예: NVIDIA가 들어간 기사 ──
df_NVIDIA = search('NVIDIA')
_=df_NVIDIA['article_id'].apply(show_article, chars=500)

## 9. 문제가 있는 피드

마지막 요청이 정상(`ok`, `not_modified`)이 아닌 피드와, 연속 오류가 있는 피드를 보여 준다. 비어 있으면 모든 피드가 정상이다.

In [ ]:
# ── 9. 문제가 있는 피드 ──
problems = query("""
    SELECT s.feed_id, f.source, s.last_status, s.consecutive_errors, s.last_checked_utc, s.next_due_utc
    FROM feed_state s
    LEFT JOIN feeds f ON f.feed_id = s.feed_id
    WHERE s.last_status NOT IN ('ok', 'not_modified') OR s.consecutive_errors > 0
    ORDER BY s.consecutive_errors DESC
""")
for col in ['last_checked_utc', 'next_due_utc']:
    problems[col] = to_kst(problems[col])
problems

In [ ]:
# ── 9. 최근 24시간 피드 요청 결과 ──
# 최근 24시간 피드 요청 결과 요약 (ok, not_modified가 대부분이어야 한다)
query("""
    SELECT outcome AS 결과, COUNT(*) AS 요청수, MAX(http_status) AS HTTP_예
    FROM fetch_log
    WHERE checked_utc >= :since
    GROUP BY outcome ORDER BY 요청수 DESC
""", since=utc_ago(24))